# CAPM & Factor Models

From-scratch derivation and implementation of the Capital Asset Pricing Model, beta estimation, and multi-factor models.

**Outline**
1. The big question: what return should you expect?
2. From portfolio theory to CAPM
3. The Capital Market Line (CML)
4. The Security Market Line (SML)
5. CML vs SML: what is the difference?
6. Beta: the single most important number
7. Estimating beta from data
8. Systematic vs unsystematic risk
9. CAPM applications: cost of equity
10. Alpha: beating the market
11. The single-index model
12. Beyond CAPM: Fama-French 3-factor model
13. Factor model estimation and diagnostics
14. Summary of key concepts
15. ReferencesThis notebook covers the complete CFA Level 1 CAPM and factor models curriculum. Every concept is built up from intuition, with worked examples before code and interpretation after.

> **Prerequisites:** You should be comfortable with portfolio return/risk calculations (see the Mean-Variance notebook) and basic regression (see the Linear Regression notebook).


---
## 1. The Big Question: What Return Should You Expect?

Imagine you are considering buying shares of a company. How do you decide what return you *should* expect? Clearly a risky tech startup should offer a higher expected return than a stable utility company -- otherwise, why take the extra risk?

But how much higher? Is there a formula that tells you the "fair" expected return for any asset based on its risk?

**Yes. That formula is the CAPM.**

The Capital Asset Pricing Model (Sharpe 1964, Lintner 1965) says:

$$E[R_i] = R_f + \beta_i (E[R_M] - R_f)$$

**In plain English:** the expected return on any asset equals the risk-free rate plus a risk premium. The risk premium depends on just ONE thing: the asset's **beta** ($\beta$) -- its sensitivity to the overall market.

### Why Only One Thing?

Because in equilibrium, investors are fully diversified (they hold the market portfolio). The only risk they care about is the risk that CANNOT be diversified away -- **systematic risk** (market risk). Beta measures exactly this.

An asset's total risk includes both systematic and unsystematic (company-specific) components. But since unsystematic risk can be eliminated by diversification, the market does not reward investors for bearing it.

> **Key Concept:** The CAPM is the equilibrium counterpart to Markowitz's portfolio theory. Markowitz tells you how to build an optimal portfolio; CAPM tells you what asset prices (and expected returns) should be in equilibrium when everyone follows Markowitz.

> **CFA Exam Tip:** The CAPM equation $E[R_i] = R_f + \beta_i (E[R_M] - R_f)$ must be memorized cold. You need to be able to calculate expected returns, find beta given other information, and identify mispriced securities (alpha).### The CAPM Answer in One Sentence

> **Key Concept:** The expected return on any asset is the risk-free rate plus a premium for bearing **systematic risk**, measured by beta:
> $$E(R_i) = R_f + \beta_i [E(R_m) - R_f]$$

This deceptively simple formula is one of the most important in all of finance. It says:
1. You are compensated ONLY for systematic (market) risk
2. The compensation is proportional to your beta (sensitivity to the market)
3. Idiosyncratic (firm-specific) risk earns NO premium because it can be diversified away

> **CFA Exam Tip:** Know this formula cold. Every component has a name:
> - $R_f$ = risk-free rate
> - $\beta_i$ = beta of asset $i$
> - $E(R_m) - R_f$ = equity risk premium (ERP) or market risk premium
> - $\beta_i[E(R_m) - R_f]$ = the asset's risk premium
### The CAPM Assumptions

The CAPM rests on several idealised assumptions:

| Assumption | Reality |
|:---|:---|
| Investors are mean-variance optimisers | Reasonable approximation |
| All investors have identical expectations | Clearly false, but allows aggregation |
| Markets are frictionless (no taxes, no transaction costs) | Approximately true for large investors |
| All assets are tradeable and infinitely divisible | Mostly true for liquid markets |
| Investors can borrow/lend at the risk-free rate | Approximately true via Treasury rates |
| Single-period model | Limits applicability to multi-period decisions |

> **Common Mistake:** Students sometimes dismiss the CAPM because its assumptions are unrealistic. But a model doesn't need perfect assumptions to be useful — it needs to generate good predictions. The CAPM's prediction (only systematic risk is priced) has held up reasonably well empirically, even if its precise predictions about expected returns have not.


---
## 2. From Portfolio Theory to CAPM

The CAPM is the *equilibrium* counterpart to Markowitz's portfolio theory. The logic goes:

1. **All investors use mean-variance optimization** (Markowitz).
2. **They all see the same expected returns, volatilities, and correlations** (homogeneous expectations).
3. **They can all borrow/lend at the same risk-free rate.**
4. **Therefore, they all hold the SAME tangency portfolio** (from the separation theorem).
5. **In equilibrium, the tangency portfolio must be the market portfolio** (because if everyone holds it, the aggregate of all holdings IS the market).

This chain of reasoning transforms a *prescriptive* tool (Markowitz: "here is how to optimize") into a *descriptive* model (CAPM: "here is what prices must be in equilibrium").

### The CAPM Assumptions

- Investors are risk-averse, mean-variance optimizers
- Single period investment horizon
- Homogeneous expectations (everyone agrees on inputs)
- Unlimited borrowing/lending at the risk-free rate
- No taxes, transaction costs, or market frictions
- All assets are perfectly divisible and liquid

These assumptions are unrealistic, but the model is still remarkably useful as a benchmark.

> **CFA Exam Tip:** You may be asked about CAPM assumptions. The key ones to remember: homogeneous expectations, unlimited risk-free borrowing/lending, and single-period horizon. Real-world violations (taxes, transaction costs, heterogeneous beliefs) lead to modifications and multi-factor models.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 3. The Capital Market Line (CML)

The CML describes the risk-return relationship for **efficient portfolios** -- that is, portfolios that combine the risk-free asset with the market portfolio.

### The Formula

$$E[R_P] = R_f + \frac{E[R_M] - R_f}{\sigma_M} \times \sigma_P$$

where:
- $R_f$ = risk-free rate
- $E[R_M]$ = expected return on the market portfolio
- $\sigma_M$ = standard deviation of the market portfolio
- $\sigma_P$ = standard deviation of the efficient portfolio

**In words:** Expected return = risk-free rate + (market Sharpe ratio) x (portfolio risk).

### Key Properties

- The CML is a **straight line** from $R_f$ through the market portfolio in risk-return space.
- The slope is the **market Sharpe ratio**: $\frac{E[R_M] - R_f}{\sigma_M}$ -- this is the "price of risk" in the economy.
- **Only efficient portfolios lie on the CML.** Individual stocks and non-diversified portfolios lie below it.

### Worked Example

If $R_f = 3\%$, $E[R_M] = 10\%$, and $\sigma_M = 16\%$:
- Market Sharpe ratio = $(10\% - 3\%) / 16\% = 0.4375$
- An investor who wants $\sigma_P = 8\%$ (half the market risk) expects: $3\% + 0.4375 \times 8\% = 6.5\%$
- An investor who wants $\sigma_P = 24\%$ (leveraged) expects: $3\% + 0.4375 \times 24\% = 13.5\%$

> **Key Concept:** The CML applies ONLY to efficient portfolios (combinations of the risk-free asset and the market portfolio). It uses total risk ($\sigma$) as the risk measure because efficient portfolios have no unsystematic risk.

> **CFA Exam Tip:** The CML plots expected return vs. TOTAL risk ($\sigma$). Do not confuse it with the SML, which plots expected return vs. BETA. This distinction is critical and frequently tested.### The CML as a Straight Line

Recall from portfolio theory: when we combine the risk-free asset with any risky portfolio, the resulting risk-return combinations lie on a straight line. The CML is the BEST such line — it uses the **tangency portfolio** (the portfolio with the highest Sharpe ratio).

In the CAPM world, because all investors hold the same efficient portfolio, the tangency portfolio IS the market portfolio. Therefore:

$$\text{CML: } E(R_c) = R_f + \underbrace{\frac{E(R_m) - R_f}{\sigma_m}}_{\text{Market Sharpe ratio}} \times \sigma_c$$

The slope is the market's Sharpe ratio — the reward per unit of total risk available in the economy.

Let's plot the CML with a few stocks to see who lies on it and who doesn't:


In [ ]:
# Market parameters
rf = 0.03       # risk-free rate: 3%
mu_m = 0.10     # expected market return: 10%
sig_m = 0.16    # market volatility: 16%
sr_m = (mu_m - rf) / sig_m  # market Sharpe ratio

print(f"Market Sharpe ratio: {sr_m:.4f}")
print(f"Interpretation: for every 1% of volatility, the market rewards {sr_m:.2f}% of excess return")

# Plot the CML
sig_range = np.linspace(0, 0.30, 100)
cml = rf + sr_m * sig_range

fig, ax = plt.subplots()
ax.plot(sig_range * 100, cml * 100, color=PRIMARY, linewidth=2, label=f'CML (Sharpe = {sr_m:.3f})')
ax.plot(sig_m * 100, mu_m * 100, '*', color=ACCENT, markersize=15, zorder=5, label='Market portfolio')
ax.plot(0, rf * 100, 'D', color=TERTIARY, markersize=10, zorder=5, label=f'Risk-free ({rf*100:.0f}%)')

# Annotate regions
ax.annotate('Lending\n(conservative)', xy=(8, 6.5), fontsize=10, color='grey', ha='center')
ax.annotate('Borrowing\n(leveraged)', xy=(24, 13), fontsize=10, color='grey', ha='center')

ax.set_xlabel('Portfolio Std Dev (%)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Capital Market Line')
ax.legend()
ax.set_xlim(0, 32)
ax.set_ylim(0, 18)
plt.tight_layout()
plt.show()

### Interpreting the CML Plot

- **Points to the left of the market portfolio** represent investors who put some money in the risk-free asset ("lending"). They accept less risk and less return.
- **Points to the right** represent investors who borrow at $R_f$ to invest more than 100% in the market ("leveraging"). They take more risk for more return.
- **No rational investor should hold a portfolio below the CML** -- they could do better by mixing the market portfolio with the risk-free asset.> **Key Concept:** The CML equation is:
> $$E(R_c) = R_f + \frac{E(R_m) - R_f}{\sigma_m} \sigma_c$$
>
> The slope $\frac{E(R_m) - R_f}{\sigma_m}$ is the **market Sharpe ratio** — the "price of risk" in the economy. It tells you how much extra return the market offers per unit of total risk. Every efficient portfolio lies on this line.

> **CFA Exam Tip:** Only **efficient portfolios** (combinations of the risk-free asset and the market portfolio) lie on the CML. Individual stocks and inefficient portfolios lie BELOW the CML because they bear unsystematic risk that is not rewarded.


---
## 4. The Security Market Line (SML)

While the CML applies only to efficient portfolios, the **SML applies to ALL assets and portfolios** -- efficient or not. This is what makes it so powerful.

### The CAPM Equation (The SML)

$$E[R_i] = R_f + \beta_i (E[R_M] - R_f)$$

where:
- $\beta_i = \frac{\text{Cov}(R_i, R_M)}{\text{Var}(R_M)} = \frac{\rho_{i,M} \sigma_i}{\sigma_M}$
- $E[R_M] - R_f$ = the **equity risk premium** (or market risk premium)

### What the SML Tells You

The SML says that the expected return on ANY asset is determined by just three things:
1. The risk-free rate ($R_f$)
2. The equity risk premium ($E[R_M] - R_f$)
3. The asset's beta ($\beta_i$)

### Worked Example

If $R_f = 3\%$, $E[R_M] = 10\%$ (so equity risk premium = 7%), and a stock has $\beta = 1.5$:

$$E[R] = 3\% + 1.5 \times 7\% = 3\% + 10.5\% = 13.5\%$$

This stock must earn 13.5% to fairly compensate investors for its systematic risk. If it earns more, it has positive alpha (a bargain); if less, negative alpha (overpriced).

### Identifying Mispriced Securities

- **Above the SML** = positive alpha = undervalued (expected return exceeds fair compensation for risk). BUY.
- **Below the SML** = negative alpha = overvalued (expected return is insufficient for the risk). SELL or avoid.
- **On the SML** = fairly priced.

> **Key Concept:** The SML is the graphical representation of the CAPM. It plots expected return vs. beta (systematic risk). Every asset should lie on the SML in equilibrium. Deviations from the SML represent alpha -- positive or negative.

> **CFA Exam Tip:** Know how to read the SML diagram. Assets ABOVE the line have positive alpha (undervalued), assets BELOW have negative alpha (overvalued). The y-intercept is $R_f$ and the slope is the equity risk premium.### From CML to SML: The Key Insight

The CML uses total risk ($\sigma$) and only applies to efficient portfolios. But what about individual stocks? Their total risk includes diversifiable noise. The SML strips this away:

$$E(R_i) = R_f + \beta_i [E(R_m) - R_f]$$

This applies to ALL assets — efficient or not. The x-axis is beta (systematic risk), not sigma (total risk). A stock with high total risk but low beta (e.g., a biotech with company-specific risk) will have a LOW expected return according to CAPM.

Let's plot the SML and see where our simulated stocks fall:


In [ ]:
# Plot the Security Market Line with simulated stocks
betas_range = np.linspace(0, 2.0, 100)
sml = rf + betas_range * (mu_m - rf)  # the SML line

# Simulate 20 stocks with some deviation from the SML (alpha != 0)
n_stocks = 20
stock_betas = rng.uniform(0.3, 1.8, n_stocks)
stock_alphas = rng.normal(0, 0.015, n_stocks)  # some positive, some negative alpha
stock_expected_returns = rf + stock_betas * (mu_m - rf) + stock_alphas

fig, ax = plt.subplots()
ax.plot(betas_range, sml * 100, color=PRIMARY, linewidth=2, label='SML')

# Color stocks by alpha: green = positive (undervalued), red = negative (overvalued)
scatter = ax.scatter(stock_betas, stock_expected_returns * 100, c=stock_alphas, 
           cmap='RdYlGn', s=80, edgecolors='black', zorder=5, vmin=-0.03, vmax=0.03)

ax.plot(1.0, mu_m * 100, '*', color=ACCENT, markersize=15, zorder=6, label='Market (beta=1)')
ax.set_xlabel('Beta')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Security Market Line\n(color = alpha: green = positive/undervalued, red = negative/overvalued)')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the SML Plot

- **Stocks ON the SML** are fairly priced according to CAPM -- their expected return exactly compensates for their systematic risk.
- **Green stocks (above the SML)** have positive alpha -- they offer more return than their risk warrants. These are potential buys.
- **Red stocks (below the SML)** have negative alpha -- they offer less return than their risk warrants. These are potential sells.
- **The market portfolio** (gold star) always lies exactly on the SML at $\beta = 1$.> **Key Concept:** The SML is the graphical representation of the CAPM equation. Its slope is the equity risk premium $(E(R_m) - R_f)$, and it passes through two points: $(\beta = 0, R_f)$ and $(\beta = 1, E(R_m))$.

> **CFA Exam Tip:** Stocks ABOVE the SML have positive alpha (underpriced — buy!). Stocks BELOW the SML have negative alpha (overpriced — sell!). In equilibrium, all stocks should lie ON the SML. Any deviation represents a potential investment opportunity.

**Worked Example:** If $R_f = 3\%$, $E(R_m) = 10\%$, and Stock X has $\beta = 1.4$:
$$E(R_X) = 3\% + 1.4 \times (10\% - 3\%) = 3\% + 9.8\% = 12.8\%$$
If Stock X's actual expected return is 15%, its alpha is $15\% - 12.8\% = 2.2\%$ (above the SML → buy).


---
## 5. CML vs SML: What Is the Difference?

This is a common source of confusion, so let's be very clear:

| Feature | CML | SML |
|:--|:--|:--|
| **X-axis** | Total risk ($\sigma$) | Systematic risk ($\beta$) |
| **Applies to** | Efficient portfolios only | ALL assets and portfolios |
| **What it shows** | How to combine Rf + market | Fair pricing of any asset |
| **Slope** | Market Sharpe ratio | Equity risk premium |
| **Use case** | Asset allocation decisions | Security valuation |

### The Key Distinction

- **CML:** "If I am willing to take X% total risk, what return can I earn with an efficient portfolio?"
- **SML:** "Given this asset's systematic risk (beta), what return SHOULD I expect?"

### Why Two Different Lines?

The CML uses total risk because for efficient portfolios, ALL risk is systematic (unsystematic risk has been diversified away). The SML uses beta because for individual assets, only systematic risk is priced -- the market does not reward you for unsystematic risk because you could have diversified it away.

> **CFA Exam Tip:** This distinction is almost guaranteed to appear on the exam. Remember: CML = $\sigma$ on x-axis = efficient portfolios only. SML = $\beta$ on x-axis = everything. If a question asks about individual stock pricing, you want the SML.| Feature | CML | SML |
|:---|:---|:---|
| **X-axis** | Total risk ($\sigma$) | Systematic risk ($\beta$) |
| **Applies to** | Efficient portfolios only | ALL assets and portfolios |
| **Slope** | Market Sharpe ratio | Equity risk premium |
| **Intercept** | $R_f$ | $R_f$ |
| **Key use** | Determining the optimal portfolio | Pricing individual securities |

> **CFA Exam Tip:** The CML is for **portfolio allocation** (how much risk to take). The SML is for **security valuation** (is this stock fairly priced for its risk?). Don't confuse them — they have different x-axes!


---
## 6. Beta: The Single Most Important Number

Beta deserves a deep dive because it is central to everything in the CAPM.

### Three Ways to Think About Beta

**1. Sensitivity interpretation:** Beta measures how much an asset's return moves when the market moves by 1%.
- $\beta = 1.0$: moves exactly with the market
- $\beta = 1.5$: moves 50% MORE than the market (if market goes up 10%, stock goes up ~15%)
- $\beta = 0.5$: moves 50% LESS than the market
- $\beta = 0$: no market sensitivity (like cash)
- $\beta < 0$: moves OPPOSITE to the market (very rare for stocks)

**2. Statistical interpretation:** Beta is the slope of the regression of asset returns on market returns:

$$R_i - R_f = \alpha_i + \beta_i (R_M - R_f) + \epsilon_i$$

This regression line is called the **characteristic line**. Its slope is beta, its intercept is alpha.

**3. Risk contribution interpretation:** Beta tells you how much systematic risk an asset contributes to your portfolio. A stock with $\beta = 2$ contributes twice as much market risk as a stock with $\beta = 1$.

### The Beta Formula

$$\beta_i = \frac{\text{Cov}(R_i, R_M)}{\text{Var}(R_M)} = \rho_{i,M} \frac{\sigma_i}{\sigma_M}$$

**In words:** beta = (how correlated the asset is with the market) x (how volatile it is relative to the market).

### Worked Example

A stock has $\sigma_i = 30\%$, $\rho_{i,M} = 0.6$, and the market has $\sigma_M = 16\%$:

$$\beta = 0.6 \times \frac{30\%}{16\%} = 0.6 \times 1.875 = 1.125$$

This stock is slightly more sensitive to market movements than the average stock.

### Typical Beta Values by Sector

| Sector | Typical Beta | Intuition |
|:--|:--|:--|
| Utilities | 0.5 - 0.7 | Stable demand, regulated |
| Consumer Staples | 0.6 - 0.8 | People always buy food |
| Healthcare | 0.8 - 1.0 | Somewhat defensive |
| S&P 500 | 1.0 | By definition |
| Technology | 1.2 - 1.5 | Growth-sensitive |
| Biotech | 1.4 - 1.8 | High uncertainty |

> **Key Concept:** Beta measures systematic risk -- the risk that cannot be diversified away. It is the ONLY risk that matters for determining expected return in the CAPM framework.

> **CFA Exam Tip:** Beta can be calculated from the formula $\beta = \text{Cov}(R_i, R_M) / \text{Var}(R_M)$. You may also see it expressed as $\beta = \rho_{i,M} \sigma_i / \sigma_M$. Both are equivalent. Make sure you can compute beta from either form.### Beta Values and Their Meaning

| Beta | Meaning | Example |
|:---:|:---|:---|
| $\beta < 0$ | Moves opposite to market (very rare) | Gold miners (sometimes) |
| $\beta = 0$ | No market sensitivity | Risk-free asset |
| $0 < \beta < 1$ | Less volatile than market | Utilities, consumer staples |
| $\beta = 1$ | Moves with market | Market index, average stock |
| $\beta > 1$ | More volatile than market | Tech stocks, small caps, financials |
| $\beta > 2$ | Very aggressive | Leveraged positions, speculative stocks |

### What Beta Is NOT

> **Common Mistake:** Beta is NOT a measure of total risk. A stock with $\beta = 0.5$ and $\sigma = 40\%$ has LOW systematic risk but HIGH total risk (lots of idiosyncratic volatility). Beta only captures the portion of risk that moves with the market.

> **Key Concept:** Beta can be thought of as a regression slope:
> $$\beta_i = \frac{\text{Cov}(R_i, R_m)}{\text{Var}(R_m)} = \rho_{i,m} \frac{\sigma_i}{\sigma_m}$$
> It depends on three things: the correlation with the market ($\rho$), the stock's volatility ($\sigma_i$), and the market's volatility ($\sigma_m$).


---
## 7. Estimating Beta from Data

In theory, beta is defined in terms of expected returns. In practice, we estimate it from *historical* returns using **ordinary least squares (OLS) regression**.

### The Regression

$$R_{i,t} - R_{f,t} = \alpha + \beta (R_{M,t} - R_{f,t}) + \epsilon_t$$

- **Dependent variable:** stock excess return
- **Independent variable:** market excess return
- **Slope ($\beta$):** estimated beta
- **Intercept ($\alpha$):** Jensen's alpha (excess return beyond CAPM prediction)
- **$R^2$:** fraction of return variance explained by the market (= proportion of systematic risk)

### Practical Considerations

- **Time period:** Industry standard is 60 months (5 years) of monthly returns, or 2 years of weekly returns.
- **Market proxy:** Usually the S&P 500 or a broad market index.
- **Standard error:** Beta estimates are noisy -- a standard error of 0.2-0.3 is common, so the 95% confidence interval for a stock with $\hat{\beta} = 1.3$ might be [0.7, 1.9].
- **Mean reversion:** Estimated betas tend to revert toward 1.0 over time, which is why Bloomberg adjusts: $\beta_{\text{adj}} = \frac{2}{3} \hat{\beta} + \frac{1}{3} (1.0)$.

> **CFA Exam Tip:** Know that beta is estimated via regression of excess stock returns on excess market returns. The slope is beta, the intercept is alpha, and $R^2$ tells you what fraction of total risk is systematic.

Let's estimate beta for a simulated stock with known parameters to see how the process works.### Practical Issues in Beta Estimation

| Choice | Options | Impact |
|:---|:---|:---|
| **Return frequency** | Daily, weekly, monthly | Daily → more data but more noise; monthly → less noise but fewer observations |
| **Estimation period** | 1yr, 2yr, 5yr | Longer → more stable but may miss regime changes |
| **Market index** | S&P 500, MSCI World, local index | Choice matters for international stocks |
| **Adjusted beta** | Raw vs Blume adjustment | Blume: $\beta_{adj} = \frac{2}{3}\beta_{raw} + \frac{1}{3}(1.0)$ — shrinks toward 1.0 |

> **CFA Exam Tip:** The Blume adjustment is used by Bloomberg and many practitioners. The rationale: betas tend to **mean-revert** toward 1.0 over time (extreme betas moderate). The adjustment formula $\frac{2}{3}\beta + \frac{1}{3}$ captures this tendency.

> **Common Mistake:** Using daily returns with a 5-year window gives 1,260 data points — seems great. But daily returns are noisy, and thin trading can bias beta estimates downward for small/illiquid stocks. Many practitioners prefer monthly returns over 5 years (60 observations) as a compromise.


In [ ]:
def estimate_beta(stock_returns, market_returns, rf_rate=0.0):
    """Estimate CAPM beta via OLS regression.
    
    Regresses stock excess returns on market excess returns:
        R_i - Rf = alpha + beta * (R_M - Rf) + epsilon
    
    Returns: beta, alpha, r_squared, std_err_beta
    """
    excess_stock = stock_returns - rf_rate
    excess_market = market_returns - rf_rate
    
    # Manual OLS (no libraries needed)
    x_bar = np.mean(excess_market)
    y_bar = np.mean(excess_stock)
    
    # Beta = Cov(stock, market) / Var(market)
    beta = np.sum((excess_market - x_bar) * (excess_stock - y_bar)) / np.sum((excess_market - x_bar)**2)
    alpha = y_bar - beta * x_bar
    
    # R-squared: how much of stock variance is explained by the market
    y_hat = alpha + beta * excess_market
    residuals = excess_stock - y_hat
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((excess_stock - y_bar)**2)
    r_squared = 1 - ss_res / ss_tot
    
    # Standard error of beta (for hypothesis testing)
    n = len(stock_returns)
    se_beta = np.sqrt(ss_res / (n - 2) / np.sum((excess_market - x_bar)**2))
    
    return beta, alpha, r_squared, se_beta

# --- Simulate a stock with known parameters ---
T = 60          # 60 monthly returns (standard for beta estimation)
true_beta = 1.3
true_alpha = 0.002  # 20 basis points per month of outperformance
sigma_eps = 0.04    # idiosyncratic volatility

# Generate market returns and stock returns from the factor model
market_returns = rng.normal(mu_m / 12, sig_m / np.sqrt(12), T)
stock_returns = (true_alpha + true_beta * (market_returns - rf / 12) 
                 + rng.normal(0, sigma_eps, T) + rf / 12)

# Estimate beta
beta_hat, alpha_hat, r2, se_beta = estimate_beta(stock_returns, market_returns, rf / 12)

print(f"True beta:       {true_beta:.3f}")
print(f"Estimated beta:  {beta_hat:.3f} +/- {se_beta:.3f}")
print(f"95% CI:          [{beta_hat - 1.96*se_beta:.3f}, {beta_hat + 1.96*se_beta:.3f}]")
print(f"Estimated alpha: {alpha_hat*100:.3f}% per month ({alpha_hat*12*100:.2f}% annualized)")
print(f"R-squared:       {r2:.3f} ({r2*100:.1f}% of return variance explained by market)")

### Interpreting the Results

- **Estimated beta** is close to the true value (1.3) but not exact -- this is estimation error from having only 60 observations.
- **The 95% confidence interval** is wide, reflecting the inherent uncertainty in beta estimates.
- **R-squared** tells us what fraction of the stock's return variance is explained by market movements. The remainder is idiosyncratic (stock-specific) risk.
- **Alpha** is the return earned beyond what CAPM predicts -- positive alpha means outperformance.> **Key Concept:** The regression $R_i - R_f = \alpha + \beta(R_m - R_f) + \varepsilon$ is called the **characteristic line** or **market model**. The slope is beta, the intercept is alpha, and the residuals $\varepsilon$ represent idiosyncratic returns. The $R^2$ of this regression tells you what fraction of the stock's return variation is explained by the market — i.e., the proportion of systematic risk.


In [ ]:
# Plot the characteristic line (regression of excess returns)
excess_m = market_returns - rf / 12
excess_s = stock_returns - rf / 12

fig, ax = plt.subplots()
ax.scatter(excess_m * 100, excess_s * 100, alpha=0.6, color=PRIMARY, edgecolors='white', s=60)

# Regression line
x_line = np.linspace(excess_m.min(), excess_m.max(), 100)
y_line = alpha_hat + beta_hat * x_line
ax.plot(x_line * 100, y_line * 100, color=SECONDARY, linewidth=2, 
        label=f'beta = {beta_hat:.2f}, alpha = {alpha_hat*100:.2f}%/month')

ax.axhline(0, color='grey', alpha=0.3)
ax.axvline(0, color='grey', alpha=0.3)
ax.set_xlabel('Market Excess Return (%)')
ax.set_ylabel('Stock Excess Return (%)')
ax.set_title(f'Characteristic Line (R-squared = {r2:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

### Reading the Characteristic Line

- **The slope** of the line is beta -- a steeper line means higher market sensitivity.
- **The intercept** (where the line crosses the y-axis) is alpha -- positive means the stock outperforms CAPM predictions.
- **The scatter** around the line represents idiosyncratic risk (the $\epsilon$ term). More scatter = more unsystematic risk = lower $R^2$.> **Key Concept:** The characteristic line is the visual representation of the market model regression. Key features to identify:
> - **Slope** = beta (steeper = more sensitive to market)
> - **Y-intercept** = alpha (above zero = outperformance)
> - **Scatter around the line** = idiosyncratic risk (more scatter = more unsystematic risk)
> - **$R^2$** = how tightly the points cluster around the line (higher = more systematic risk as a fraction of total)

> **CFA Exam Tip:** If given a scatter plot of stock excess returns vs market excess returns, you should be able to estimate beta visually (the slope), alpha (the y-intercept), and the relative amounts of systematic vs unsystematic risk (how tightly points cluster around the line).


---
## 8. Systematic vs Unsystematic Risk

This is one of the most important concepts in all of finance. Every asset's total risk can be split into two components:

$$\sigma_i^2 = \underbrace{\beta_i^2 \sigma_M^2}_{\text{Systematic}} + \underbrace{\sigma_{\epsilon_i}^2}_{\text{Unsystematic}}$$

### Systematic Risk (Market Risk, Non-Diversifiable)

- Caused by economy-wide factors: recessions, interest rate changes, inflation, geopolitical events.
- Affects ALL assets (though to different degrees, depending on beta).
- **Cannot be eliminated by diversification.**
- **This is the risk the market rewards you for bearing.**

### Unsystematic Risk (Idiosyncratic, Diversifiable, Company-Specific)

- Caused by company-specific events: CEO departure, product recall, lawsuit, earnings surprise.
- Affects only one company or a small group.
- **CAN be eliminated by holding a diversified portfolio.**
- **The market does NOT reward you for bearing this risk** -- you chose not to diversify, so that is your problem.

### The Diversification Effect

As you add more stocks to a portfolio:
- Unsystematic risk shrinks toward zero (the law of large numbers -- individual company shocks cancel out).
- Systematic risk remains as a "floor" that you cannot diversify below.
- With 20-30 randomly selected stocks, you eliminate roughly 90% of unsystematic risk.

> **Key Concept:** Only systematic risk is rewarded with higher expected return. This is WHY beta (which measures systematic risk) is the only risk factor in the CAPM. Total risk ($\sigma$) includes unsystematic risk, which the market does not compensate.

> **CFA Exam Tip:** If asked "why does CAPM use beta instead of standard deviation?", the answer is: because only systematic risk is priced. Standard deviation includes unsystematic risk, which investors can diversify away for free.### The Risk Decomposition

Total risk = Systematic risk + Unsystematic risk

$$\sigma_i^2 = \beta_i^2 \sigma_m^2 + \sigma_{\varepsilon}^2$$

In terms of the regression: $R^2 = \frac{\beta^2 \sigma_m^2}{\sigma_i^2}$ is the fraction that is systematic.

> **Key Concept:** This decomposition is the theoretical justification for diversification and for the CAPM pricing result:
> - Unsystematic risk can be eliminated by holding many assets → no premium
> - Systematic risk cannot be diversified away → must be compensated with higher expected return

> **CFA Exam Tip:** If asked "why does the CAPM only compensate systematic risk?", the answer is: because unsystematic risk can be diversified away for free. Rational investors hold diversified portfolios, so they only bear systematic risk. Therefore, only systematic risk matters for pricing.


In [ ]:
# Decompose the simulated stock's risk
total_var = np.var(stock_returns)
systematic_var = beta_hat**2 * np.var(market_returns)
unsystematic_var = total_var - systematic_var

print("Risk Decomposition:")
print(f"  Total variance:        {total_var*10000:.2f} bps-squared")
print(f"  Systematic variance:   {systematic_var*10000:.2f} ({systematic_var/total_var*100:.1f}% of total)")
print(f"  Unsystematic variance: {unsystematic_var*10000:.2f} ({unsystematic_var/total_var*100:.1f}% of total)")
print(f"\n  Note: systematic fraction ({systematic_var/total_var*100:.1f}%) should match R-squared ({r2*100:.1f}%)")

### Reading the Risk Decomposition

The output shows the variance decomposition: what fraction of total variance is systematic ($\beta^2 \sigma_m^2$) and what fraction is idiosyncratic ($\sigma_\varepsilon^2$). A stock with $R^2 = 0.40$ has 40% systematic risk and 60% idiosyncratic risk.

> **Key Concept:** For a well-diversified portfolio, idiosyncratic risk virtually disappears, and only the systematic component remains. This is why CAPM prices only the systematic portion.


### The Diversification Effect Visualized

Let's see what happens as you add more stocks to a portfolio. Unsystematic risk shrinks toward zero, but systematic risk remains as the floor.We'll create portfolios with increasing numbers of stocks (each with the same beta and idiosyncratic risk) and watch the total portfolio volatility decline toward the systematic risk floor:


In [ ]:
# Demonstrate: portfolio of N stocks -- systematic risk is the floor
n_stocks_range = np.arange(1, 51)
avg_idio_var = 0.04**2     # average idiosyncratic variance per stock
avg_beta = 1.0             # average beta
systematic = avg_beta**2 * (sig_m / np.sqrt(12))**2  # monthly systematic variance

# In an equally-weighted portfolio of N stocks, idiosyncratic variance scales as 1/N
portfolio_var = systematic + avg_idio_var / n_stocks_range

fig, ax = plt.subplots()
ax.plot(n_stocks_range, np.sqrt(portfolio_var) * 100 * np.sqrt(12), 
        color=PRIMARY, linewidth=2, label='Total risk')
ax.axhline(np.sqrt(systematic) * 100 * np.sqrt(12), color=SECONDARY, 
           linestyle='--', linewidth=2, label='Systematic risk (the floor)')
ax.fill_between(n_stocks_range, 
                np.sqrt(systematic) * 100 * np.sqrt(12), 
                np.sqrt(portfolio_var) * 100 * np.sqrt(12), 
                alpha=0.2, color=TERTIARY, label='Unsystematic (diversifiable)')
ax.set_xlabel('Number of Stocks')
ax.set_ylabel('Annualized Volatility (%)')
ax.set_title('Diversification and Risk Reduction')
ax.legend()
plt.tight_layout()
plt.show()

### Reading the Plot

- **With 1 stock,** you bear all the idiosyncratic risk plus market risk.
- **As you add stocks,** idiosyncratic risk shrinks (the green area gets thinner).
- **By 20-30 stocks,** most idiosyncratic risk is gone.
- **The dashed line** is the systematic risk floor -- you cannot go below this no matter how many stocks you hold.

This is the visual proof of why only systematic risk matters: since you can cheaply eliminate unsystematic risk by diversifying, the market does not reward you for bearing it.> **Key Concept:** The curve flattens dramatically after about 20-30 stocks. This is why:
> - An index fund with 500 stocks is not dramatically less risky than one with 50
> - The remaining risk (the floor) is **market risk** — it affects ALL stocks
> - This floor is $\beta_p^2 \sigma_m^2$, where $\beta_p \to 1$ for a broad portfolio


---
## 9. CAPM Applications: Cost of Equity

One of the most practical applications of CAPM is estimating the **cost of equity** -- the return that shareholders require to hold a company's stock. This is critical for:

- **Corporate finance:** DCF valuation uses cost of equity as the discount rate.
- **Capital budgeting:** Companies use it to decide which projects to undertake.
- **Regulation:** Utilities' allowed return on equity is often set using CAPM.

### The Formula

$$k_e = R_f + \beta_e (E[R_M] - R_f)$$

This is just the CAPM equation applied to a specific company. The equity risk premium ($E[R_M] - R_f$) is typically estimated at 5-7% based on historical data.

### Worked Example

For a technology company with $\beta = 1.35$, if $R_f = 3\%$ and the equity risk premium is 7%:

$$k_e = 3\% + 1.35 \times 7\% = 3\% + 9.45\% = 12.45\%$$

This means the company must earn at least 12.45% on its equity investments to satisfy shareholders.

> **CFA Exam Tip:** The cost of equity calculation using CAPM is one of the most frequently tested applications. Be comfortable plugging in numbers and interpreting the result.### CAPM in Corporate Finance

The cost of equity is a critical input to the **Weighted Average Cost of Capital (WACC)**:

$$\text{WACC} = w_e \times k_e + w_d \times k_d(1 - t)$$

where $k_e$ is estimated via CAPM. A higher beta → higher $k_e$ → higher WACC → higher discount rate for projects → fewer projects accepted.

**Worked Example:** A company with $\beta = 1.2$, $R_f = 4\%$, ERP $= 6\%$:
$$k_e = 4\% + 1.2 \times 6\% = 11.2\%$$

If the company is evaluating a project with expected return of 10%, it should REJECT it (10% < 11.2% = required return).

> **CFA Exam Tip:** The CFA exam frequently asks you to compute cost of equity using CAPM and use it to make accept/reject decisions for capital projects. Remember: the appropriate discount rate depends on the PROJECT's beta, not the company's beta (if they differ in risk).
Let's compute the cost of equity for several sectors and see how beta drives required returns:


In [ ]:
# Cost of equity for different sectors using CAPM
sectors = {
    'Utilities':        0.55,
    'Consumer Staples': 0.70,
    'Healthcare':       0.85,
    'S&P 500 (Market)': 1.00,
    'Industrials':      1.15,
    'Technology':       1.35,
    'Biotech':          1.60,
}

equity_risk_premium = mu_m - rf  # 7%

print(f"Assumptions: Rf = {rf*100:.0f}%, Equity Risk Premium = {equity_risk_premium*100:.0f}%")
print(f"\n{'Sector':<20} {'Beta':>6} {'Cost of Equity':>15}")
print('-' * 43)
for sector, beta in sectors.items():
    ke = rf + beta * equity_risk_premium
    print(f"{sector:<20} {beta:>6.2f} {ke*100:>14.2f}%")

print(f"\nInterpretation: a Biotech company needs to earn {rf*100 + 1.6*equity_risk_premium*100:.1f}%")
print(f"to compensate shareholders for the risk, while a Utility needs only {rf*100 + 0.55*equity_risk_premium*100:.2f}%.")

---
## 10. Alpha: Beating the Market

**Jensen's alpha** is the holy grail of active management. It measures the return an investment generates *beyond* what the CAPM predicts for its level of systematic risk.

### The Formula

$$\alpha = R_P - [R_f + \beta_P (R_M - R_f)]$$

- $\alpha > 0$: the manager generated **excess return** beyond CAPM. This is "skill" (or luck).
- $\alpha = 0$: the manager earned exactly what CAPM predicts. No value added.
- $\alpha < 0$: the manager **underperformed** relative to CAPM. The investor would have been better off in an index fund.

### Why Alpha Is So Important

- **For active managers:** Alpha justifies their fees. If $\alpha \leq 0$ after fees, investors should switch to a passive index fund.
- **For investors:** Alpha helps distinguish skilled managers from those riding market beta.
- **For the efficient market hypothesis:** If markets are efficient, alpha should be zero on average (after costs).

### The Harsh Reality

Academic research consistently finds that:
- The average mutual fund has **negative alpha** after fees.
- Only about 2-5% of managers show persistent positive alpha.
- Most apparent "alpha" is explained by exposure to additional risk factors (size, value, momentum).

This is one reason passive investing has grown so rapidly.

> **Key Concept:** Alpha is the residual return after accounting for market risk exposure. It is the definitive measure of active management skill.

> **CFA Exam Tip:** Know Jensen's alpha formula and be able to calculate it. A positive alpha means the portfolio plots ABOVE the SML; negative alpha means BELOW the SML.### Testing for Alpha: Is It Real?

The statistical significance of alpha matters enormously. A fund with $\alpha = 2\%$ annually might just be noise:

**The t-test:** $t_\alpha = \frac{\hat{\alpha}}{SE(\hat{\alpha})}$

With monthly data and typical volatility, you need roughly **20+ years** of data to detect a 1% annual alpha with statistical significance. This is why the active vs passive debate is so difficult to resolve empirically.

> **Key Concept:** Most observed "alpha" is likely noise or exposure to unpriced risk factors rather than genuine skill. The Fama-French 3-factor model explains much of what appears to be alpha under the single-factor CAPM. Adding more factors (momentum, quality, low volatility) explains even more.
### The Active vs Passive Debate

The difficulty of generating statistically significant alpha is the core argument for passive (index) investing:

| Evidence | Implication |
|:---|:---|
| ~80% of active equity funds underperform their benchmark over 15 years (SPIVA) | Most managers lack skill after fees |
| Alpha that survives CAPM often disappears under FF3 | "Skill" may be factor exposure |
| 20+ years needed to statistically confirm 1% annual alpha | Investors can't wait that long |
| Fees (1-2% for active) vs ETFs (0.03-0.10%) | Fee drag overwhelms modest alpha |


---
## 11. The Single-Index Model

Estimating a full covariance matrix for $N$ assets requires $N(N+1)/2$ parameters. For 500 stocks, that is 125,250 parameters -- far too many to estimate reliably from historical data.

The **single-index model** (Sharpe 1963) simplifies this dramatically by assuming all correlations between stocks arise from their common exposure to the market:

$$R_i = \alpha_i + \beta_i R_M + \epsilon_i$$

where $\epsilon_i$ is independent across stocks.

### The Simplification

Under this model, the covariance between any two stocks is:

$$\text{Cov}(R_i, R_j) = \beta_i \beta_j \sigma_M^2$$

The entire covariance matrix is determined by just $3N + 1$ parameters ($\alpha_i$, $\beta_i$, $\sigma_{\epsilon_i}$ for each stock, plus $\sigma_M^2$). For 500 stocks: 1,501 parameters instead of 125,250.

### The Tradeoff

- **Advantage:** Far fewer parameters to estimate, so estimates are more stable.
- **Disadvantage:** The model assumes all correlation between stocks comes through the market factor. Industry-specific correlations (e.g., two oil stocks moving together beyond market movements) are ignored.

> **Key Concept:** The single-index model trades accuracy for stability. It captures the dominant source of correlation (the market) while ignoring industry and other group effects. For large portfolios, this tradeoff is often worthwhile.### Why the Single-Index Model Matters

For a portfolio of $N$ assets, the full covariance matrix has $N(N+1)/2$ unique entries. The single-index model replaces this with just $3N + 1$ parameters:
- $N$ betas, $N$ alphas, $N$ residual variances, plus 1 market variance

| $N$ assets | Full covariance | Single-index model | Reduction |
|:---:|:---:|:---:|:---:|
| 10 | 55 | 31 | 44% |
| 50 | 1,275 | 151 | 88% |
| 500 | 125,250 | 1,501 | 99% |

> **Key Concept:** The single-index model makes portfolio optimisation practical for large universes. It assumes all correlations between stocks come through their common exposure to the market factor: $\sigma_{ij} = \beta_i \beta_j \sigma_m^2$. This is a strong assumption, but it works surprisingly well in practice.

> **CFA Exam Tip:** The single-index model is also called the "market model" or "diagonal model" (because the covariance matrix, after removing the market factor, is diagonal — only variances remain, no covariances).


In [ ]:
# Demonstrate: single-index vs full covariance estimation
n_assets = 10
T_sim = 120  # 10 years of monthly data

# True parameters
betas_true = rng.uniform(0.5, 1.5, n_assets)
alphas_true = rng.normal(0, 0.001, n_assets)
sigma_eps_true = rng.uniform(0.02, 0.06, n_assets)

# Generate returns from the single-index model
r_market = rng.normal(mu_m / 12, sig_m / np.sqrt(12), T_sim)
returns = np.zeros((T_sim, n_assets))
for i in range(n_assets):
    returns[:, i] = alphas_true[i] + betas_true[i] * r_market + rng.normal(0, sigma_eps_true[i], T_sim)

# Method 1: Full sample covariance (N*(N+1)/2 = 55 parameters)
cov_full = np.cov(returns.T)

# Method 2: Single-index covariance (3*N + 1 = 31 parameters)
betas_est = np.array([estimate_beta(returns[:, i], r_market)[0] for i in range(n_assets)])
var_m = np.var(r_market)
residuals = returns - np.outer(r_market, betas_est)
var_eps = np.var(residuals, axis=0)
cov_index = np.outer(betas_est, betas_est) * var_m + np.diag(var_eps)

# Visualize both
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
im1 = ax1.imshow(cov_full * 10000, cmap='Blues')
ax1.set_title('Full Sample Covariance\n(55 free parameters)')
plt.colorbar(im1, ax=ax1, label='(bps-squared)')

im2 = ax2.imshow(cov_index * 10000, cmap='Blues')
ax2.set_title('Single-Index Covariance\n(31 free parameters)')
plt.colorbar(im2, ax=ax2, label='(bps-squared)')

plt.tight_layout()
plt.show()

print(f"Frobenius norm of difference: {np.linalg.norm(cov_full - cov_index)*10000:.2f} bps-squared")
print(f"The two matrices are similar -- the single-index model captures the main structure.")

---
## 12. Beyond CAPM: The Fama-French 3-Factor Model

Empirical research has found that the CAPM does not fully explain stock returns. Some patterns that CAPM cannot explain:

### The Size Effect
Small-cap stocks tend to outperform large-cap stocks, even after adjusting for beta. This suggests that "size" is a separate risk factor.

### The Value Effect
Stocks with high book-to-market ratios ("value stocks") tend to outperform low book-to-market stocks ("growth stocks"), even after adjusting for beta and size.

### The Fama-French 3-Factor Model

To capture these patterns, Fama and French (1993) proposed:

$$E[R_i] - R_f = \beta_{i,\text{MKT}} (E[R_M] - R_f) + \beta_{i,\text{SMB}} \cdot E[\text{SMB}] + \beta_{i,\text{HML}} \cdot E[\text{HML}]$$

where:
- **MKT** = market excess return (same as CAPM)
- **SMB** = "Small Minus Big" -- return of small-cap stocks minus large-cap stocks
- **HML** = "High Minus Low" -- return of value stocks minus growth stocks

### What This Means in Practice

A stock's expected return depends on THREE exposures, not just one:
1. **Market beta** ($\beta_{\text{MKT}}$): sensitivity to overall market
2. **Size beta** ($\beta_{\text{SMB}}$): sensitivity to the small-cap premium
3. **Value beta** ($\beta_{\text{HML}}$): sensitivity to the value premium

### Example Interpretation

If a stock has $\beta_{\text{MKT}} = 1.1$, $\beta_{\text{SMB}} = 0.6$, $\beta_{\text{HML}} = -0.3$:
- It is slightly more sensitive to the market than average.
- It behaves like a **small-cap stock** (positive SMB loading).
- It behaves like a **growth stock** (negative HML loading -- opposite of value).

> **Key Concept:** The Fama-French model says that CAPM's single factor (market) is not enough. Size and value are additional priced risk factors. Much of what looks like "alpha" in the CAPM is actually compensation for exposure to these additional factors.

> **CFA Exam Tip:** Know that the Fama-French model extends CAPM with SMB and HML. Understand that what appears as alpha in CAPM may be explained by factor exposures in a multi-factor model.### The Three Factors Explained

| Factor | Abbreviation | What it captures | Long/Short construction |
|:---|:---|:---|:---|
| **Market** | MKT-RF | Broad equity risk | Long market, short T-bills |
| **Size** | SMB (Small Minus Big) | Small-cap premium | Long small stocks, short large stocks |
| **Value** | HML (High Minus Low) | Value premium | Long high B/M stocks, short low B/M stocks |

The 3-factor model:
$$R_i - R_f = \alpha_i + \beta_i^{MKT}(R_m - R_f) + \beta_i^{SMB} \cdot SMB + \beta_i^{HML} \cdot HML + \varepsilon_i$$

> **Key Concept:** If a fund's "alpha" under CAPM disappears when you add SMB and HML, the manager wasn't generating true alpha — they were just tilting toward small-cap and/or value stocks (which anyone can do cheaply with index funds).

### Beyond Fama-French: The Factor Zoo

Since 1993, researchers have identified hundreds of "factors." The most widely accepted additions:

| Factor | What it captures |
|:---|:---|
| **Momentum** (WML) | Stocks that went up continue to go up (short-term) |
| **Profitability** (RMW) | Firms with high operating profitability outperform |
| **Investment** (CMA) | Firms that invest conservatively outperform aggressive investors |
| **Low volatility** | Low-risk stocks earn higher risk-adjusted returns (anomaly!) |

> **CFA Exam Tip:** The CFA curriculum covers the Fama-French model as an extension of CAPM. Know that: (1) it explains returns better than CAPM alone (higher $R^2$), (2) it attributes some CAPM "alpha" to size and value exposures, (3) it doesn't fully explain all return patterns (momentum is the biggest gap).


---
## 13. Factor Model Estimation and Diagnostics

Let's estimate the Fama-French 3-factor model from simulated data and compare it to CAPM. We will see how adding factors improves the model's explanatory power.We'll estimate both the single-factor CAPM and the Fama-French 3-factor model on the same simulated stock, then compare $R^2$, alpha, and factor exposures. The key question: does adding size and value factors improve the model?

> **What to watch for:** The FF3 model should have a higher $R^2$ (explains more return variation) and potentially a different alpha (what appeared to be skill under CAPM might just be factor exposure).


In [ ]:
# Simulate Fama-French factors (monthly)
T_ff = 120  # 10 years monthly
mkt_excess = rng.normal(0.005, 0.045, T_ff)  # market excess return
smb = rng.normal(0.002, 0.03, T_ff)           # small-cap premium
hml = rng.normal(0.003, 0.03, T_ff)           # value premium

# Create a stock with KNOWN factor exposures
true_betas_ff = {'MKT': 1.1, 'SMB': 0.6, 'HML': -0.3}
true_alpha_ff = 0.001  # 10 bps monthly alpha

# Generate stock returns from the 3-factor model
stock_excess = (true_alpha_ff 
                + true_betas_ff['MKT'] * mkt_excess 
                + true_betas_ff['SMB'] * smb 
                + true_betas_ff['HML'] * hml 
                + rng.normal(0, 0.025, T_ff))

def ols_regression(y, X):
    """Manual OLS: y = [1, X] @ beta + epsilon.
    
    Returns: coefficients, standard errors, R-squared, adjusted R-squared
    """
    X_aug = np.column_stack([np.ones(len(y)), X])  # add intercept
    beta = np.linalg.lstsq(X_aug, y, rcond=None)[0]
    y_hat = X_aug @ beta
    residuals = y - y_hat
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r_squared = 1 - ss_res / ss_tot
    n, k = X_aug.shape
    r_squared_adj = 1 - (1 - r_squared) * (n - 1) / (n - k)
    mse = ss_res / (n - k)
    se = np.sqrt(np.diag(mse * np.linalg.inv(X_aug.T @ X_aug)))
    return beta, se, r_squared, r_squared_adj

# --- CAPM regression (1 factor) ---
beta_capm, se_capm, r2_capm, _ = ols_regression(stock_excess, mkt_excess.reshape(-1, 1))

# --- Fama-French 3-factor regression ---
X_ff = np.column_stack([mkt_excess, smb, hml])
beta_ff, se_ff, r2_ff, r2_adj_ff = ols_regression(stock_excess, X_ff)

print("=== CAPM Regression (1 factor) ===")
print(f"  Alpha:    {beta_capm[0]*100:.3f}% monthly (SE: {se_capm[0]*100:.3f}%)")
print(f"  beta_MKT: {beta_capm[1]:.3f} (SE: {se_capm[1]:.3f})")
print(f"  R-sq:     {r2_capm:.3f}")

print("\n=== Fama-French 3-Factor Regression ===")
factor_names = ['Alpha', 'beta_MKT', 'beta_SMB', 'beta_HML']
true_values = [true_alpha_ff, true_betas_ff['MKT'], true_betas_ff['SMB'], true_betas_ff['HML']]
for name, b, se, true_val in zip(factor_names, beta_ff, se_ff, true_values):
    sig = '***' if abs(b/se) > 2.576 else '**' if abs(b/se) > 1.96 else '*' if abs(b/se) > 1.645 else ''
    if name == 'Alpha':
        print(f"  {name:<10}: {b*100:.3f}% monthly (SE: {se*100:.3f}%) [true: {true_val*100:.1f}%] {sig}")
    else:
        print(f"  {name:<10}: {b:.3f} (SE: {se:.3f}) [true: {true_val:.1f}] {sig}")
print(f"  R-sq:     {r2_ff:.3f} (vs CAPM: {r2_capm:.3f})")
print(f"  Adj R-sq: {r2_adj_ff:.3f}")
print(f"\nAdding SMB and HML improved R-squared by {(r2_ff - r2_capm)*100:.1f} percentage points.")

### Interpreting the Results

- The CAPM R-squared is lower because it tries to explain returns with only one factor (market).
- The FF3 model captures more of the return variation by adding size and value factors.
- The estimated factor loadings are close to the true values, with confidence intervals containing the truth.
- The improvement in $R^2$ shows that size and value are genuine sources of return variation, not just noise.> **Key Concept:** The jump in $R^2$ from CAPM to FF3 tells you how much of the stock's return was driven by size and value factors rather than the broad market. If a "stock picker" fund shows high CAPM alpha but zero FF3 alpha, the manager was really just buying small-cap value stocks — a strategy that can be replicated cheaply with ETFs.

> **Common Mistake:** A higher $R^2$ doesn't mean a better fund. It means the model explains more of the return. A fund with $R^2 = 0.99$ and $\alpha = 0$ is just tracking factors passively. A fund with $R^2 = 0.70$ and significant $\alpha = 3\%$ is generating genuine excess returns.


In [ ]:
# Visualize factor exposures: true vs estimated
fig, ax = plt.subplots(figsize=(8, 5))
factors = ['MKT', 'SMB', 'HML']
true_vals = [true_betas_ff['MKT'], true_betas_ff['SMB'], true_betas_ff['HML']]
est_vals = beta_ff[1:]   # skip intercept
est_se = se_ff[1:]

x = np.arange(len(factors))
w = 0.3
ax.bar(x - w/2, true_vals, w, label='True', color=PRIMARY, edgecolor='white')
ax.bar(x + w/2, est_vals, w, label='Estimated', color=SECONDARY, edgecolor='white',
       yerr=1.96 * est_se, capsize=5)  # 95% confidence intervals
ax.set_xticks(x)
ax.set_xticklabels(factors)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Factor Loading')
ax.set_title('Fama-French Factor Exposures: True vs Estimated')
ax.legend()
plt.tight_layout()
plt.show()

### Residual Diagnostics

A good model should have residuals that are:
1. **Normally distributed** (for valid inference on alpha and betas)
2. **Uncorrelated over time** (no patterns left unexplained)
3. **Homoskedastic** (constant variance)

Let's check.> **Important:** If residuals show autocorrelation (today's residual predicts tomorrow's), the standard errors of $\alpha$ and $\beta$ are understated — you might think alpha is significant when it isn't. Use **Newey-West** standard errors to correct for this. If residuals are heteroscedastic (variance changes over time), use **White** robust standard errors.


In [ ]:
# Residual diagnostics for the FF3 model
X_aug = np.column_stack([np.ones(T_ff), X_ff])
residuals_ff = stock_excess - X_aug @ beta_ff

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Histogram of residuals (should look normal)
axes[0].hist(residuals_ff * 100, bins=25, color=PRIMARY, edgecolor='white', density=True)
x_norm = np.linspace(residuals_ff.min() * 100, residuals_ff.max() * 100, 100)
axes[0].plot(x_norm, stats.norm.pdf(x_norm, 0, residuals_ff.std() * 100), color=SECONDARY, linewidth=2)
axes[0].set_xlabel('Residual (%)')
axes[0].set_title('Residual Distribution')

# 2. Q-Q plot (dots should lie on the line)
sorted_res = np.sort(residuals_ff)
theoretical = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_res)))
axes[1].scatter(theoretical, sorted_res * 100, alpha=0.5, color=PRIMARY, s=20)
fit_line = np.polyfit(theoretical, sorted_res * 100, 1)
axes[1].plot(theoretical, np.polyval(fit_line, theoretical), color=SECONDARY, linewidth=2)
axes[1].set_xlabel('Theoretical Quantiles')
axes[1].set_ylabel('Sample Quantiles (%)')
axes[1].set_title('Q-Q Plot')

# 3. Autocorrelation (bars should be within the dashed lines)
lags = np.arange(1, 13)
acf = [np.corrcoef(residuals_ff[:-l], residuals_ff[l:])[0, 1] for l in lags]
axes[2].bar(lags, acf, color=PRIMARY, edgecolor='white')
axes[2].axhline(1.96 / np.sqrt(T_ff), color=SECONDARY, linestyle='--', alpha=0.7)
axes[2].axhline(-1.96 / np.sqrt(T_ff), color=SECONDARY, linestyle='--', alpha=0.7)
axes[2].set_xlabel('Lag')
axes[2].set_ylabel('Autocorrelation')
axes[2].set_title('Residual ACF')

plt.tight_layout()
plt.show()

---
## 14. Summary of Key Concepts

| Concept | Key Idea |
|:--|:--|
| **CAPM** | Expected return = $R_f + \beta(E[R_M] - R_f)$ -- only systematic risk is rewarded |
| **CML** | Risk-return line for *efficient portfolios*; x-axis = $\sigma$; slope = Sharpe ratio |
| **SML** | Risk-return line for *all assets*; x-axis = $\beta$; slope = equity risk premium |
| **Beta** | Sensitivity to market = $\text{Cov}(R_i, R_M) / \text{Var}(R_M)$ |
| **Alpha** | Return beyond CAPM prediction = skill (or luck) |
| **Systematic risk** | Market risk; cannot be diversified away; measured by $\beta^2 \sigma_M^2$ |
| **Unsystematic risk** | Company-specific risk; diversifiable; not compensated |
| **Single-index model** | Simplifies covariance estimation using one market factor |
| **Fama-French 3-factor** | Extends CAPM with size (SMB) and value (HML) factors |### CAPM Limitations and the Real World

| Criticism | Evidence | Implication |
|:---|:---|:---|
| Beta alone doesn't explain returns | Fama-French showed size and value matter | Multi-factor models needed |
| Beta is unstable over time | Estimated betas change significantly | Use adjusted beta or rolling estimates |
| Risk-free borrowing rate unrealistic | Investors can't borrow at T-bill rates | Zero-beta version of CAPM |
| Single-period model | Real investment is multi-period | Intertemporal CAPM (Merton) |
| Market portfolio is unobservable | We use proxies (S&P 500) | Roll's critique — CAPM is untestable |

Despite these limitations, CAPM remains the most widely used model for estimating expected returns in corporate finance and investment management. Its simplicity and intuitive appeal are powerful advantages.

### The CAPM Formula Card

| Concept | Formula |
|:---|:---|
| CAPM | $E(R_i) = R_f + \beta_i[E(R_m) - R_f]$ |
| Beta | $\beta_i = \text{Cov}(R_i, R_m) / \text{Var}(R_m)$ |
| Blume adjusted beta | $\beta_{adj} = \frac{2}{3}\beta + \frac{1}{3}$ |
| CML | $E(R_c) = R_f + \frac{E(R_m) - R_f}{\sigma_m}\sigma_c$ |
| Jensen's alpha | $\alpha_i = \bar{R}_i - [R_f + \beta_i(\bar{R}_m - R_f)]$ |
| Risk decomposition | $\sigma_i^2 = \beta_i^2\sigma_m^2 + \sigma_{\varepsilon,i}^2$ |
| FF3 model | $R_i - R_f = \alpha + \beta^{MKT}MKT + \beta^{SMB}SMB + \beta^{HML}HML + \varepsilon$ |


---
## 15. References

1. Sharpe, W. F. "Capital Asset Prices," *Journal of Finance*, 1964.
2. Lintner, J. "The Valuation of Risk Assets," *Review of Economics and Statistics*, 1965.
3. Fama, E. F. & French, K. R. "Common Risk Factors in the Returns on Stocks and Bonds," *JFE*, 1993.
4. Bodie, Z., Kane, A., Marcus, A. *Investments*, 12th ed., McGraw-Hill, 2021.
5. CFA Institute, *CFA Program Curriculum Level I -- Portfolio Management*.### CFA Level 1 Curriculum Alignment

Key Learning Outcome Statements covered:
- LOS: Describe the capital asset pricing model (CAPM) and its assumptions
- LOS: Calculate and interpret the expected return using the CAPM
- LOS: Describe the capital market line and the security market line
- LOS: Calculate and interpret beta
- LOS: Describe systematic and unsystematic risk
- LOS: Explain the use of factor models in return attribution
